#1. Título y objetivo del proyecto

- Nombre del proyecto:
- Presentado por: Juan Carlos Salazar Chávez
- Objetivo: construir un sistema RAG con dataset real, retrieval, reranker, citas, grounding y evaluación

#1. Instalación de librerías

In [ ]:
!pip install --upgrade --force-reinstall transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.0/802.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36

In [1]:
!pip -q install wikipedia-api pandas requests tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 4.7 MB/s eta 0:00:00


In [2]:
import requests
import pandas as pd
import wikipediaapi
from tqdm import tqdm

#2. Cargamos el dataset

###Archivos de Wikipedia

In [3]:
WIKI_API_URL = "https://es.wikipedia.org/w/api.php"

def buscar_titulos_wikipedia(termino, limite=20):
    params = {
        "action": "query",
        "list": "search",
        "srsearch": termino,
        "format": "json",
        "srlimit": limite
    }
    response = requests.get(WIKI_API_URL, params=params)
    data = response.json()
    resultados = data.get("query", {}).get("search", [])
    return [r["title"] for r in resultados]

In [4]:
semillas = [
    # --- MINERÍA GENERAL ---
    "minería",
    "minería en Perú",
    "industria minera",
    "actividad minera",
    "recursos minerales",

    # --- MINERALES ---
    "oro",
    "cobre",
    "plata",
    "zinc",
    "hierro",
    "minerales metálicos",
    "minerales no metálicos",

    # --- PROCESOS MINEROS ---
    "extracción minera",
    "procesamiento de minerales",
    "concentración de minerales",
    "flotación de minerales",
    "lixiviación",
    "refinación de metales",
    "fundición de metales",

    # --- TIPOS DE MINERÍA ---
    "minería a cielo abierto",
    "minería subterránea",
    "minería artesanal",
    "minería ilegal",

    # --- HIDROCARBUROS ---
    "hidrocarburos",
    "petróleo",
    "gas natural",
    "industria petrolera",
    "industria del gas",

    # --- PROCESOS PETROLEROS ---
    "exploración petrolera",
    "perforación petrolera",
    "producción de petróleo",
    "refinación de petróleo",
    "transporte de hidrocarburos",

    # --- GAS NATURAL ---
    "gas natural licuado",
    "GNL",
    "gasoducto",
    "exportación de gas natural",

    # --- ECONOMÍA Y REGULACIÓN ---
    "canon minero",
    "regalías mineras",
    "contratos petroleros",
    "concesión minera",
    "impacto ambiental de la minería",
    "impacto ambiental del petróleo",

    # --- CONTEXTO PERÚ ---
    "minería en el Perú",
    "hidrocarburos en el Perú",
    "sector energético peruano",
    "Camisea",
    "exportaciones mineras Perú",

    # --- SEGURIDAD Y MEDIO AMBIENTE ---
    "pasivos ambientales mineros",
    "remediación ambiental",
    "seguridad minera",
    "contaminación minera",
    "derrames de petróleo"
]

In [5]:
import requests
import time

WIKI_API_URL = "https://es.wikipedia.org/w/api.php"

def buscar_titulos_wikipedia(termino, limite=20, pausa=1):
    params = {
        "action": "query",
        "list": "search",
        "srsearch": termino,
        "format": "json",
        "srlimit": limite
    }

    headers = {
        "User-Agent": "ProyectoRAG/1.0 (Colab educational project)"
    }

    try:
        response = requests.get(WIKI_API_URL, params=params, headers=headers, timeout=20)

        # Verificar código de estado
        if response.status_code != 200:
            print(f"Error HTTP {response.status_code} para término: {termino}")
            return []

        # Verificar que realmente vino JSON
        content_type = response.headers.get("Content-Type", "")
        if "application/json" not in content_type:
            print(f"Respuesta no JSON para término: {termino}")
            print("Primeros 200 caracteres de la respuesta:")
            print(response.text[:200])
            return []

        data = response.json()
        resultados = data.get("query", {}).get("search", [])

        time.sleep(pausa)  # pequeña pausa para evitar bloqueos
        return [r["title"] for r in resultados]

    except requests.exceptions.RequestException as e:
        print(f"Error de conexión para término '{termino}': {e}")
        return []

    except ValueError as e:
        print(f"Error al decodificar JSON para término '{termino}': {e}")
        print("Primeros 200 caracteres de la respuesta:")
        print(response.text[:200] if 'response' in locals() else "Sin respuesta")
        return []

In [6]:
titulos = set()

for termino in semillas:
    encontrados = buscar_titulos_wikipedia(termino, limite=15)
    titulos.update(encontrados)

titulos = list(titulos)

print(f"Títulos encontrados: {len(titulos)}")
print(titulos[:20])

Títulos encontrados: 588
['Jean-Baptiste Élie de Beaumont', 'Arquitectura en hierro', 'Mina Quiruvilca', 'Siglo de Oro', 'Ángela Grossheim', 'Minería hidráulica', 'Crisis energética de Perú de 2026', 'José Hierro', 'Sonatrach', 'Servicio de Evaluación Ambiental de Chile', 'Lixiviación (metalurgia)', 'Virgen de la Caridad del Cobre', 'Mina de Reocín', 'Molino semiautógeno', 'Edad de los Metales', 'Origen inorgánico del petróleo', 'Bahía de Bizkaia Gas', 'Exportaciones de Bolivia en 1995', 'Metalurgia extractiva', 'Campos petroleros de Lago Agrio']


In [7]:
wiki = wikipediaapi.Wikipedia(
    language='es',
    user_agent='ProyectoRAG/1.0 (Colab)'
)

documentos = []

for titulo in tqdm(titulos):
    page = wiki.page(titulo)

    if page.exists():
        texto = page.text.strip()

        if len(texto) >= 800:
            documentos.append({
                "title": page.title,
                "text": texto,
                "url": page.fullurl,
                "n_chars": len(texto)
            })

print(f"Documentos válidos: {len(documentos)}")

100%|██████████| 588/588 [03:33<00:00,  2.75it/s]

Documentos válidos: 561


In [8]:
df_docs = pd.DataFrame(documentos).drop_duplicates(subset=["title"]).reset_index(drop=True)
df_docs["doc_id"] = ["DOC_" + str(i).zfill(4) for i in range(1, len(df_docs) + 1)]
df_docs["n_chars"] = df_docs["text"].str.len()
df_docs["n_words"] = df_docs["text"].str.split().str.len()

df_docs = df_docs[["doc_id", "title", "url", "text", "n_chars", "n_words"]].reset_index(drop=True)

print(df_docs.shape)
df_docs.head()

(561, 6)


,doc_id,title,url,text,n_chars,n_words
0,DOC_0001,Jean-Baptiste Élie de Beaumont,https://es.wikipedia.org/wiki/Jean-Baptiste_%C...,Jean-Baptiste Armand Louis Léonce Élie de Beau...,2571,392
1,DOC_0002,Arquitectura en hierro,https://es.wikipedia.org/wiki/Arquitectura_en_...,"Arquitectura en hierro, del hierro o arquitect...",12978,1981
2,DOC_0003,Mina Quiruvilca,https://es.wikipedia.org/wiki/Mina_Quiruvilca,La mina Quiruvilca es un yacimiento minero de ...,6353,1025
3,DOC_0004,Siglo de Oro,https://es.wikipedia.org/wiki/Siglo_de_Oro,El Siglo de Oro español es un periodo históric...,48048,7771
4,DOC_0005,Ángela Grossheim,https://es.wikipedia.org/wiki/%C3%81ngela_Gros...,Ángela María del Rosario Grossheim Barrientos ...,1588,249


In [9]:
print(f"Total de documentos: {len(df_docs)}")

Total de documentos: 561


In [10]:
df_docs.to_csv("mineria_hidrocarburos_dataset.csv", index=False, encoding="utf-8")

In [11]:
df_docs[["title", "text", "url", "n_chars"]].sample(10, random_state=42)

,title,text,url,n_chars
513,Patrones de sustitución en hidrocarburos aromá...,Los patrones de sustitución en hidrocarburos a...,https://es.wikipedia.org/wiki/Patrones_de_sust...,3333
342,Mineral del Monte,Esta página se refiere a una localidad. Para e...,https://es.wikipedia.org/wiki/Mineral_del_Monte,20825
177,Historia del petróleo en Venezuela,Venezuela es el país con las mayores reservas ...,https://es.wikipedia.org/wiki/Historia_del_pet...,33252
86,Ley que regula los pasivos ambientales de la a...,La Ley que regula los pasivos ambientales de l...,https://es.wikipedia.org/wiki/Ley_que_regula_l...,2489
333,Asesinato de Alfredo Vracko,El asesinato de Alfredo Vracko ocurrió el 19 d...,https://es.wikipedia.org/wiki/Asesinato_de_Alf...,7174
140,Hierro,El hierro​​ es un elemento químico de número a...,https://es.wikipedia.org/wiki/Hierro,29368
321,Edad del Cobre,"La edad de cobre, también llamada Calcolítico ...",https://es.wikipedia.org/wiki/Edad_del_Cobre,42263
531,Biosorción,Se denomina biosorción a un proceso fisicoquím...,https://es.wikipedia.org/wiki/Biosorci%C3%B3n,6784
101,Río de la Plata,El Río de la Plata​ está formado por la unión ...,https://es.wikipedia.org/wiki/R%C3%ADo_de_la_P...,20971
519,Gas de alumbrado,Gas industrial que se obtiene a partir de la h...,https://es.wikipedia.org/wiki/Gas_de_alumbrado,8627


In [12]:
df_docs = df_docs.drop_duplicates(subset=["title"]).reset_index(drop=True)

df_docs["doc_id"] = ["DOC_" + str(i).zfill(4) for i in range(1, len(df_docs) + 1)]
df_docs["n_chars"] = df_docs["text"].str.len()
df_docs["n_words"] = df_docs["text"].str.split().str.len()

print(df_docs.shape)
df_docs.head()

(561, 6)


,doc_id,title,url,text,n_chars,n_words
0,DOC_0001,Jean-Baptiste Élie de Beaumont,https://es.wikipedia.org/wiki/Jean-Baptiste_%C...,Jean-Baptiste Armand Louis Léonce Élie de Beau...,2571,392
1,DOC_0002,Arquitectura en hierro,https://es.wikipedia.org/wiki/Arquitectura_en_...,"Arquitectura en hierro, del hierro o arquitect...",12978,1981
2,DOC_0003,Mina Quiruvilca,https://es.wikipedia.org/wiki/Mina_Quiruvilca,La mina Quiruvilca es un yacimiento minero de ...,6353,1025
3,DOC_0004,Siglo de Oro,https://es.wikipedia.org/wiki/Siglo_de_Oro,El Siglo de Oro español es un periodo históric...,48048,7771
4,DOC_0005,Ángela Grossheim,https://es.wikipedia.org/wiki/%C3%81ngela_Gros...,Ángela María del Rosario Grossheim Barrientos ...,1588,249


In [13]:
print("Total documentos:", len(df_docs))

Total documentos: 561


#3. Chunking+Overlap

In [14]:
def chunk_text(text, chunk_size=500, overlap=100):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = words[start:end]
        chunks.append(" ".join(chunk))

        if end >= len(words):
            break

        start = end - overlap

    return chunks

###Probamos la función con los documentos

In [15]:
chunks_data = []

for _, row in df_docs.iterrows():
    chunks = chunk_text(row["text"], chunk_size=500, overlap=100)

    for j, chunk in enumerate(chunks, start=1):
        chunks_data.append({
            "chunk_id": f"{row['doc_id']}_CHUNK_{str(j).zfill(3)}",
            "doc_id": row["doc_id"],
            "title": row["title"],
            "url": row["url"],
            "chunk_order": j,
            "chunk_text": chunk,
            "n_words": len(chunk.split())
        })

df_chunks = pd.DataFrame(chunks_data)

print(df_chunks.shape)
df_chunks.head()

(2696, 7)


,chunk_id,doc_id,title,url,chunk_order,chunk_text,n_words
0,DOC_0001_CHUNK_001,DOC_0001,Jean-Baptiste Élie de Beaumont,https://es.wikipedia.org/wiki/Jean-Baptiste_%C...,1,Jean-Baptiste Armand Louis Léonce Élie de Beau...,392
1,DOC_0002_CHUNK_001,DOC_0002,Arquitectura en hierro,https://es.wikipedia.org/wiki/Arquitectura_en_...,1,"Arquitectura en hierro, del hierro o arquitect...",500
2,DOC_0002_CHUNK_002,DOC_0002,Arquitectura en hierro,https://es.wikipedia.org/wiki/Arquitectura_en_...,2,"Dockyard (Bermudas, Edward Holl, años 1820) se...",500
3,DOC_0002_CHUNK_003,DOC_0002,Arquitectura en hierro,https://es.wikipedia.org/wiki/Arquitectura_en_...,3,que también demostró las posibilidades de los ...,500
4,DOC_0002_CHUNK_004,DOC_0002,Arquitectura en hierro,https://es.wikipedia.org/wiki/Arquitectura_en_...,4,"hierro Los constructores ""de hierro"" (cast-iro...",500


In [16]:
df_chunks.to_csv("chunks_mineria_hidrocarburos.csv", index=False, encoding="utf-8")
print("Chunks guardados.")

Chunks guardados.


#4. Embeddings

###Instalamos e importamos librerías necesarias

In [17]:
!pip -q install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 87.8 MB/s eta 0:00:00


In [18]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

###Cargamos el modelo de embeddings

In [19]:
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(embedding_model_name)

print("Modelo de embeddings cargado.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo de embeddings cargado.


###Preparamos los textos de los chunks

In [20]:
chunk_texts = df_chunks["chunk_text"].tolist()
print("Total de chunks:", len(chunk_texts))

Total de chunks: 2696


###Generamos los embeddings

In [21]:
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Shape embeddings:", chunk_embeddings.shape)

Batches:   0%|          | 0/85 [00:00<?, ?it/s]

Shape embeddings: (2696, 384)


##Guardamos los embeddings

In [22]:
np.save("chunk_embeddings_mineria_hidrocarburos.npy", chunk_embeddings)
print("Embeddings guardados.")

Embeddings guardados.


#5. FAISS IHNSW

##5.1 FAISS

###Convertimos a float32

In [23]:
embeddings_faiss = chunk_embeddings.astype("float32")
dimension = embeddings_faiss.shape[1]

print("Dimensión:", dimension)

Dimensión: 384


##5.2 HNSW

In [24]:
M = 32
index = faiss.IndexHNSWFlat(dimension, M)

index.hnsw.efConstruction = 200
index.hnsw.efSearch = 64

index.add(embeddings_faiss)

print("Índice FAISS HNSW construido.")
print("Total vectores indexados:", index.ntotal)

Índice FAISS HNSW construido.
Total vectores indexados: 2696


###Guardamos el índice

In [25]:
faiss.write_index(index, "faiss_hnsw_mineria_hidrocarburos.bin")
print("Índice guardado.")

Índice guardado.


#6. RETRIEVAL

###Crear función de recuperación

In [26]:
def retrieve_chunks(query, embedding_model, index, df_chunks, top_k=5):
    query_vector = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = index.search(query_vector, top_k)

    resultados = df_chunks.iloc[indices[0]].copy()
    resultados["score_distance"] = distances[0]

    return resultados

###Probar con una pregunta del dominio

In [27]:
consulta = "¿Qué es la minería?"
resultados = retrieve_chunks(
    query=consulta,
    embedding_model=embedding_model,
    index=index,
    df_chunks=df_chunks,
    top_k=5
)

resultados[["chunk_id", "title", "url", "score_distance"]]

,chunk_id,title,url,score_distance
1851,DOC_0379_CHUNK_001,Minero,https://es.wikipedia.org/wiki/Minero,10.919001
1409,DOC_0281_CHUNK_001,Mina (minería),https://es.wikipedia.org/wiki/Mina_(miner%C3%ADa),12.925301
1678,DOC_0347_CHUNK_001,Minería,https://es.wikipedia.org/wiki/Miner%C3%ADa,13.516574
1971,DOC_0401_CHUNK_001,Liberación mineral,https://es.wikipedia.org/wiki/Liberaci%C3%B3n_...,13.779055
2308,DOC_0475_CHUNK_001,Pozo (minería),https://es.wikipedia.org/wiki/Pozo_(miner%C3%ADa),14.560461


###Ver el contenido recuperado

In [28]:
for i, row in resultados.iterrows():
    print("=" * 80)
    print("CHUNK_ID:", row["chunk_id"])
    print("TITLE   :", row["title"])
    print("URL  :", row["url"])
    print("DIST    :", row["score_distance"])
    print("\nTEXTO:\n")
    print(row["chunk_text"][:1200])
    print("\n")

CHUNK_ID: DOC_0379_CHUNK_001
TITLE   : Minero
URL  : https://es.wikipedia.org/wiki/Minero
DIST    : 10.919000625610352

TEXTO:

Se denomina minero a la persona que se encarga de excavar minas para extraer minerales. Las principales ocupaciones de un minero incluyen taladrar la roca con picos y palas o utilizando herramientas eléctricas para extraer el mineral, apuntalar los túneles con soportes de madera para impedir su derrumbe, desplegar vías para el transporte de la piedra o cargar el mineral en vagonetas para su transporte al exterior. En ocasiones, los mineros realizan funciones auxiliares como crear túneles de pasaje o ventilación, excavar salas o pozos para facilitar la actividad de extracción.​ El trabajo de un minero en el interior de la mina es duro. En primer lugar, están privados de la luz del sol por lo que deben alumbrarse con lámparas acopladas a sus cascos. En segundo lugar, se trata de un trabajo sucio pues el polvo de mineral impregna las ropas, el cabello y la piel d

In [29]:
print(chunk_embeddings.shape)
print(index.ntotal)
resultados[["chunk_id", "title", "url", "score_distance"]]

(2696, 384)
2696


,chunk_id,title,url,score_distance
1851,DOC_0379_CHUNK_001,Minero,https://es.wikipedia.org/wiki/Minero,10.919001
1409,DOC_0281_CHUNK_001,Mina (minería),https://es.wikipedia.org/wiki/Mina_(miner%C3%ADa),12.925301
1678,DOC_0347_CHUNK_001,Minería,https://es.wikipedia.org/wiki/Miner%C3%ADa,13.516574
1971,DOC_0401_CHUNK_001,Liberación mineral,https://es.wikipedia.org/wiki/Liberaci%C3%B3n_...,13.779055
2308,DOC_0475_CHUNK_001,Pozo (minería),https://es.wikipedia.org/wiki/Pozo_(miner%C3%ADa),14.560461


#7. RERANKER

In [30]:
!pip -q install sentence-transformers

In [31]:
from sentence_transformers import CrossEncoder

reranker_model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(reranker_model_name)

print("Reranker cargado correctamente.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Reranker cargado correctamente.


###Función para reordenar los chunks recuperados

In [32]:
def rerank_chunks(query, resultados_faiss, reranker):
    pares = [(query, row["chunk_text"]) for _, row in resultados_faiss.iterrows()]
    scores = reranker.predict(pares)

    resultados_rerank = resultados_faiss.copy()
    resultados_rerank["rerank_score"] = scores

    resultados_rerank = resultados_rerank.sort_values(
        by="rerank_score",
        ascending=False
    ).reset_index(drop=True)

    return resultados_rerank

###Probamos FAISS + reranker

In [33]:
pregunta = "¿Qué es la minería?"

resultados_faiss = retrieve_chunks(
    query=pregunta,
    embedding_model=embedding_model,
    index=index,
    df_chunks=df_chunks,
    top_k=8
)

print("=== RESULTADOS FAISS ===")
display(resultados_faiss[["chunk_id", "title", "url", "score_distance"]])

resultados_rerank = rerank_chunks(
    query=pregunta,
    resultados_faiss=resultados_faiss,
    reranker=reranker
)

print("=== RESULTADOS RERANKEADOS ===")
display(resultados_rerank[["chunk_id", "title", "url", "score_distance", "rerank_score"]])

=== RESULTADOS FAISS ===


,chunk_id,title,url,score_distance
1851,DOC_0379_CHUNK_001,Minero,https://es.wikipedia.org/wiki/Minero,10.919001
1409,DOC_0281_CHUNK_001,Mina (minería),https://es.wikipedia.org/wiki/Mina_(miner%C3%ADa),12.925301
1678,DOC_0347_CHUNK_001,Minería,https://es.wikipedia.org/wiki/Miner%C3%ADa,13.516574
1971,DOC_0401_CHUNK_001,Liberación mineral,https://es.wikipedia.org/wiki/Liberaci%C3%B3n_...,13.779055
2308,DOC_0475_CHUNK_001,Pozo (minería),https://es.wikipedia.org/wiki/Pozo_(miner%C3%ADa),14.560461
2689,DOC_0556_CHUNK_001,Mina a cielo abierto,https://es.wikipedia.org/wiki/Mina_a_cielo_abi...,15.637810
229,DOC_0045_CHUNK_001,Minería artesanal,https://es.wikipedia.org/wiki/Miner%C3%ADa_art...,15.743779
2065,DOC_0420_CHUNK_001,Minería de remoción de cima,https://es.wikipedia.org/wiki/Miner%C3%ADa_de_...,15.901548


=== RESULTADOS RERANKEADOS ===


,chunk_id,title,url,score_distance,rerank_score
0,DOC_0347_CHUNK_001,Minería,https://es.wikipedia.org/wiki/Miner%C3%ADa,13.516574,5.178763
1,DOC_0045_CHUNK_001,Minería artesanal,https://es.wikipedia.org/wiki/Miner%C3%ADa_art...,15.743779,4.860977
2,DOC_0556_CHUNK_001,Mina a cielo abierto,https://es.wikipedia.org/wiki/Mina_a_cielo_abi...,15.637810,4.584486
3,DOC_0420_CHUNK_001,Minería de remoción de cima,https://es.wikipedia.org/wiki/Miner%C3%ADa_de_...,15.901548,4.482923
4,DOC_0401_CHUNK_001,Liberación mineral,https://es.wikipedia.org/wiki/Liberaci%C3%B3n_...,13.779055,1.404474
5,DOC_0281_CHUNK_001,Mina (minería),https://es.wikipedia.org/wiki/Mina_(miner%C3%ADa),12.925301,1.249929
6,DOC_0379_CHUNK_001,Minero,https://es.wikipedia.org/wiki/Minero,10.919001,0.008186
7,DOC_0475_CHUNK_001,Pozo (minería),https://es.wikipedia.org/wiki/Pozo_(miner%C3%ADa),14.560461,-0.870683


###Vemos los mejores chunks ya rerankeados

In [34]:
for i, row in resultados_rerank.head(4).iterrows():
    print("=" * 80)
    print("RANK:", i + 1)
    print("CHUNK_ID:", row["chunk_id"])
    print("TITLE   :", row["title"])
    print("URL  :", row["url"])
    print("FAISS DIST:", row["score_distance"])
    print("RERANK SCORE:", row["rerank_score"])
    print("\nTEXTO:\n")
    print(row["chunk_text"][:1200])
    print("\n")

RANK: 1
CHUNK_ID: DOC_0347_CHUNK_001
TITLE   : Minería
URL  : https://es.wikipedia.org/wiki/Miner%C3%ADa
FAISS DIST: 13.516573905944824
RERANK SCORE: 5.178763389587402

TEXTO:

La minería es una actividad económica del sector primario cuando nos referimos a la extracción de minerales, y del sector energético si hacemos referencia a la extracción de combustibles fósiles. Consiste en la explotación o extracción de los minerales. Dependiendo del tipo de mineral a extraer la actividad se divide en minería metalúrgica (cobre, oro, plata, aluminio, plomo, hierro, mercurio, etc.), que son empleados como materias primas básicas; la minería no metalúrgica o también denominada de cantera y construcción (arcilla, cuarzo, zafiro, esmeralda, granito, mármol, mica, etc.) obtiene materiales de construcción y materia prima para joyería y ornamentación, entre otros usos. Otro tipo de minería es la extracción de los minerales energéticos o combustibles, empleados principalmente para generar energía, com

#8. LLM Qwen

###Importamos librerías

In [ ]:
!pip -q install -U transformers accelerate sentencepiece huggingface_hub

In [ ]:
#!pip install --upgrade --force-reinstall transformers accelerate

###Cargamos Qwen

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Modelo Qwen cargado correctamente.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Modelo Qwen cargado correctamente.


##Armamos contexto desde los mejores chunks

In [ ]:
def build_context_from_results(resultados, max_chunks=4):
    context_blocks = []

    for i, (_, row) in enumerate(resultados.head(max_chunks).iterrows(), start=1):
        bloque = (
            f"[Fuente {i}]\n"
            f"Título: {row['title']}\n"
            f"Chunk ID: {row['chunk_id']}\n"
            f"Origen: {row['url']}\n"
            f"Texto: {row['chunk_text']}\n"
        )
        context_blocks.append(bloque)

    return "\n\n".join(context_blocks)

###Función de generación con Qwen

In [ ]:
def generate_qwen_response(query, context, model, tokenizer, max_new_tokens=220):
    system_msg = (
        "Eres un asistente RAG especializado en Minería e Hidrocarburos. "
        "Responde únicamente con la información del contexto. "
        "Si la respuesta no está en el contexto, responde exactamente: "
        "'No se encontró información suficiente en el contexto recuperado.' "
        "No inventes datos."
    )

    user_msg = f"""
Pregunta:
{query}

Contexto:
{context}

Da una respuesta clara, breve y basada solo en el contexto.
""".strip()

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return response.strip()

###Pipeline completo: FAISS + reranker + Qwen

In [ ]:
def answer_with_rag_qwen_rerank(
    query,
    embedding_model,
    index,
    df_chunks,
    reranker,
    model,
    tokenizer,
    top_k_retrieval=8,
    top_k_final=4
):
    resultados_faiss = retrieve_chunks(
        query=query,
        embedding_model=embedding_model,
        index=index,
        df_chunks=df_chunks,
        top_k=top_k_retrieval
    )

    resultados_rerank = rerank_chunks(
        query=query,
        resultados_faiss=resultados_faiss,
        reranker=reranker
    )

    resultados_finales = resultados_rerank.head(top_k_final).copy()

    context = build_context_from_results(resultados_finales, max_chunks=top_k_final)

    respuesta = generate_qwen_response(query, context, model, tokenizer)

    return {
        "query": query,
        "respuesta": respuesta,
        "contexto": context,
        "resultados_faiss": resultados_faiss,
        "resultados_rerank": resultados_rerank,
        "resultados_finales": resultados_finales
    }

###Probamos una pregunta del dominio

In [ ]:
pregunta = "¿Qué es la minería?"

salida_rag = answer_with_rag_qwen_rerank(
    query=pregunta,
    embedding_model=embedding_model,
    index=index,
    df_chunks=df_chunks,
    reranker=reranker,
    model=model,
    tokenizer=tokenizer,
    top_k_retrieval=8,
    top_k_final=4
)

print("PREGUNTA:\n", salida_rag["query"])
print("\nRESPUESTA:\n", salida_rag["respuesta"])

###Vemos las fuentes finales usadas

In [ ]:
salida_rag["resultados_finales"][["chunk_id", "title", "url", "rerank_score"]]

#9. Citation per sentence

###Separamos la respuesta en oraciones

In [ ]:
import re

def split_into_sentences(text):
    text = text.strip()
    if not text:
        return []

    sentences = re.split(r'(?<=[\.\?\!])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    return sentences

###Similitud coseno

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)

    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0

    return float(np.dot(a, b) / denom)

###Asignamos fuente a cada oración

In [ ]:
def assign_citation_per_sentence(response_text, resultados_finales, embedding_model):
    sentences = split_into_sentences(response_text)

    if len(sentences) == 0:
        return []

    # embeddings de oraciones
    sentence_embeddings = embedding_model.encode(sentences, convert_to_numpy=True)

    # embeddings de chunks finales
    chunk_texts = resultados_finales["chunk_text"].tolist()
    chunk_embeddings = embedding_model.encode(chunk_texts, convert_to_numpy=True)

    cited_sentences = []

    for i, sentence in enumerate(sentences):
        sims = [cosine_similarity(sentence_embeddings[i], chunk_emb) for chunk_emb in chunk_embeddings]

        best_idx = int(np.argmax(sims))
        best_score = float(sims[best_idx])

        best_row = resultados_finales.iloc[best_idx]

        cited_sentences.append({
            "sentence": sentence,
            "source_label": f"Fuente {best_idx + 1}",
            "chunk_id": best_row["chunk_id"],
            "title": best_row["title"],
            "url": best_row["url"],
            "similarity_score": best_score
        })

    return cited_sentences

###Reconstruimos la respuesta con citas

In [ ]:
def build_cited_response(cited_sentences):
    if not cited_sentences:
        return ""

    parts = []

    for item in cited_sentences:
        parts.append(f"{item['sentence']} [{item['source_label']}]")

    return " ".join(parts)

###Probamos el pipeline actual

In [ ]:
citas = assign_citation_per_sentence(
    response_text=salida_rag["respuesta"],
    resultados_finales=salida_rag["resultados_finales"],
    embedding_model=embedding_model
)

In [ ]:
import pandas as pd

df_citas = pd.DataFrame(citas)
df_citas

In [ ]:
respuesta_citada = build_cited_response(citas)

print("RESPUESTA ORIGINAL:\n")
print(salida_rag["respuesta"])

print("\n" + "="*100 + "\n")

print("RESPUESTA CON CITAS:\n")
print(respuesta_citada)

###Integramos todo al pipeline final

In [ ]:
def answer_with_rag_qwen_rerank_citations(
    query,
    embedding_model,
    index,
    df_chunks,
    reranker,
    model,
    tokenizer,
    top_k_retrieval=8,
    top_k_final=4
):
    # 1. FAISS
    resultados_faiss = retrieve_chunks(
        query=query,
        embedding_model=embedding_model,
        index=index,
        df_chunks=df_chunks,
        top_k=top_k_retrieval
    )

    # 2. RERANK
    resultados_rerank = rerank_chunks(
        query=query,
        resultados_faiss=resultados_faiss,
        reranker=reranker
    )

    resultados_finales = resultados_rerank.head(top_k_final).copy()

    # 3. CONTEXTO
    context = build_context_from_results(resultados_finales, max_chunks=top_k_final)

    # 4. GENERACIÓN
    respuesta = generate_qwen_response(query, context, model, tokenizer)

    # 5. CITAS
    citas = assign_citation_per_sentence(
        response_text=respuesta,
        resultados_finales=resultados_finales,
        embedding_model=embedding_model
    )

    respuesta_citada = build_cited_response(citas)

    return {
        "query": query,
        "respuesta": respuesta,
        "respuesta_citada": respuesta_citada,
        "citas": citas,
        "contexto": context,
        "resultados_finales": resultados_finales
    }

###Probamos el sistema completo

In [ ]:
pregunta = "¿Qué es la minería?"

salida_final = answer_with_rag_qwen_rerank_citations(
    query=pregunta,
    embedding_model=embedding_model,
    index=index,
    df_chunks=df_chunks,
    reranker=reranker,
    model=model,
    tokenizer=tokenizer
)

print("PREGUNTA:\n", salida_final["query"])
print("\nRESPUESTA CITADA:\n", salida_final["respuesta_citada"])

#10. Hallucination Guard + Grounding Score

##Calcular grounding score

In [ ]:
def compute_grounding_score(cited_sentences):
    if not cited_sentences:
        return 0.0

    scores = [item["similarity_score"] for item in cited_sentences]
    return float(np.mean(scores))

###Clasificar el nivel de grounding

In [ ]:
def classify_grounding(grounding_score):
    if grounding_score >= 0.75:
        return "Alta sustentación"
    elif grounding_score >= 0.55:
        return "Sustentación media"
    else:
        return "Baja sustentación"

##10.1 Hallucination guard

In [ ]:
def hallucination_guard(cited_sentences, threshold=0.55):
    if not cited_sentences:
        return {
            "is_grounded": False,
            "grounding_score": 0.0,
            "label": "Baja sustentación",
            "warning": "No hay evidencia suficiente para respaldar la respuesta."
        }

    grounding_score = compute_grounding_score(cited_sentences)
    label = classify_grounding(grounding_score)

    is_grounded = grounding_score >= threshold

    if is_grounded:
        warning = "Respuesta suficientemente respaldada por el contexto recuperado."
    else:
        warning = "Posible alucinación: la respuesta no está suficientemente respaldada por el contexto recuperado."

    return {
        "is_grounded": is_grounded,
        "grounding_score": grounding_score,
        "label": label,
        "warning": warning
    }

###Probamos grounding con la salida actual

In [ ]:
diagnostico = hallucination_guard(salida_final["citas"], threshold=0.55)

diagnostico

###Mostramos grounding de forma clara

In [ ]:
print("GROUNDING SCORE:", round(diagnostico["grounding_score"], 4))
print("CLASIFICACIÓN  :", diagnostico["label"])
print("RESPALDO       :", diagnostico["is_grounded"])
print("MENSAJE        :", diagnostico["warning"])

###Integramos grounding al pipeline completo

In [ ]:
def answer_with_rag_complete(
    query,
    embedding_model,
    index,
    df_chunks,
    reranker,
    model,
    tokenizer,
    top_k_retrieval=8,
    top_k_final=4,
    grounding_threshold=0.55
):
    # 1. Retrieval inicial
    resultados_faiss = retrieve_chunks(
        query=query,
        embedding_model=embedding_model,
        index=index,
        df_chunks=df_chunks,
        top_k=top_k_retrieval
    )

    # 2. Reranking
    resultados_rerank = rerank_chunks(
        query=query,
        resultados_faiss=resultados_faiss,
        reranker=reranker
    )

    resultados_finales = resultados_rerank.head(top_k_final).copy()

    # 3. Contexto
    context = build_context_from_results(resultados_finales, max_chunks=top_k_final)

    # 4. Generación
    respuesta = generate_qwen_response(query, context, model, tokenizer)

    # 5. Citation per sentence
    citas = assign_citation_per_sentence(
        response_text=respuesta,
        resultados_finales=resultados_finales,
        embedding_model=embedding_model
    )

    respuesta_citada = build_cited_response(citas)

    # 6. Grounding / hallucination guard
    diagnostico = hallucination_guard(citas, threshold=grounding_threshold)

    return {
        "query": query,
        "respuesta": respuesta,
        "respuesta_citada": respuesta_citada,
        "citas": citas,
        "contexto": context,
        "resultados_faiss": resultados_faiss,
        "resultados_rerank": resultados_rerank,
        "resultados_finales": resultados_finales,
        "grounding_score": diagnostico["grounding_score"],
        "grounding_label": diagnostico["label"],
        "is_grounded": diagnostico["is_grounded"],
        "grounding_warning": diagnostico["warning"]
    }

###Probamos el pipeline completo final

In [ ]:
pregunta = "¿Qué es la minería?"

salida_completa = answer_with_rag_complete(
    query=pregunta,
    embedding_model=embedding_model,
    index=index,
    df_chunks=df_chunks,
    reranker=reranker,
    model=model,
    tokenizer=tokenizer,
    top_k_retrieval=8,
    top_k_final=4,
    grounding_threshold=0.55
)

print("PREGUNTA:\n", salida_completa["query"])
print("\nRESPUESTA CITADA:\n", salida_completa["respuesta_citada"])
print("\nGROUNDING SCORE:", round(salida_completa["grounding_score"], 4))
print("CLASIFICACIÓN  :", salida_completa["grounding_label"])
print("RESPALDO       :", salida_completa["is_grounded"])
print("MENSAJE        :", salida_completa["grounding_warning"])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


PREGUNTA:
 ¿Qué es la minería?

RESPUESTA CITADA:
 La minería es una actividad económica del sector primario cuando se trata de la extracción de minerales, y del sector energético si se refiere a la extracción de combustibles fósiles. [Fuente 1] Consiste en la explotación o extracción de los minerales. [Fuente 1] Según el contexto proporcionado, la minería puede dividirse en varias categorías:

1. [Fuente 1] **Minería Metalúrgica**: Extrae minerales como el cobre, oro, plata, aluminio, plomo, hierro, mercurio, etc., empleados como materias primas básicas. [Fuente 1] 2. [Fuente 2] **Minería No Metalúrgica o Mineria de Cantera y Construcción**: Extrae materiales de construcción y materia prima para joyería y ornamentación, como arcilla, cuarzo, zafiro, esmeralda, granito, mármol, mica, etc. [Fuente 3] 3. [Fuente 2] **Extracción de Minerales Energéticos o Combustibles Fósiles**: Extrae minerales como el pet [Fuente 1]

GROUNDING SCORE: 0.5802
CLASIFICACIÓN  : Sustentación media
RESPALDO  

###Ver detalle por oración

In [ ]:
pd.DataFrame(salida_completa["citas"])

,sentence,source_label,chunk_id,title,url,similarity_score
0,La minería es una actividad económica del sect...,Fuente 1,DOC_0287_CHUNK_001,Minería,https://es.wikipedia.org/wiki/Miner%C3%ADa,0.944508
1,Consiste en la explotación o extracción de los...,Fuente 1,DOC_0287_CHUNK_001,Minería,https://es.wikipedia.org/wiki/Miner%C3%ADa,0.799483
2,"Según el contexto proporcionado, la minería pu...",Fuente 1,DOC_0287_CHUNK_001,Minería,https://es.wikipedia.org/wiki/Miner%C3%ADa,0.596278
3,**Minería Metalúrgica**: Extrae minerales como...,Fuente 1,DOC_0287_CHUNK_001,Minería,https://es.wikipedia.org/wiki/Miner%C3%ADa,0.712952
4,2.,Fuente 2,DOC_0060_CHUNK_001,Minería artesanal,https://es.wikipedia.org/wiki/Miner%C3%ADa_art...,0.186198
5,**Minería No Metalúrgica o Mineria de Cantera ...,Fuente 3,DOC_0297_CHUNK_001,Mina a cielo abierto,https://es.wikipedia.org/wiki/Mina_a_cielo_abi...,0.637683
6,3.,Fuente 2,DOC_0060_CHUNK_001,Minería artesanal,https://es.wikipedia.org/wiki/Miner%C3%ADa_art...,0.202797
7,**Extracción de Minerales Energéticos o Combus...,Fuente 1,DOC_0287_CHUNK_001,Minería,https://es.wikipedia.org/wiki/Miner%C3%ADa,0.561480


###Agregar una respuesta protegida

In [ ]:
def protected_response(output):
    if output["is_grounded"]:
        return output["respuesta_citada"]
    else:
        return (
            "Advertencia: la respuesta generada presenta baja sustentación en el contexto recuperado.\n\n"
            + output["respuesta_citada"]
        )

In [ ]:
print(protected_response(salida_completa))

#Evaluación automática con ROUGE y BLEU

###Instalar librerías de evaluación

In [ ]:
!pip -q install rouge-score nltk

In [ ]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
import pandas as pd
import numpy as np

nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

###Creamos funciones de evaluación

##ROUGE

In [ ]:
def compute_rouge_scores(reference, generated):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    scores = scorer.score(reference, generated)

    return {
        "rouge1_f": scores["rouge1"].fmeasure,
        "rouge2_f": scores["rouge2"].fmeasure,
        "rougeL_f": scores["rougeL"].fmeasure
    }

##BLEU

In [ ]:
def compute_bleu_score(reference, generated):
    smoothie = SmoothingFunction().method1

    reference_tokens = reference.split()
    generated_tokens = generated.split()

    bleu = sentence_bleu(
        [reference_tokens],
        generated_tokens,
        smoothing_function=smoothie
    )

    return bleu

###Creamos el conjunto de evaluación manual

In [ ]:
evaluation_set = [
    {
        "question": "¿Qué es la minería?",
        "reference_answer": (
            "La minería es la actividad económica dedicada a la extracción de minerales "
            "desde la tierra para su uso industrial y comercial."
        )
    },
    {
        "question": "¿Qué es el petróleo crudo?",
        "reference_answer": (
            "El petróleo crudo es un recurso natural líquido compuesto por hidrocarburos, "
            "utilizado como materia prima para producir combustibles y otros derivados."
        )
    },
    {
        "question": "¿Qué es el gas natural?",
        "reference_answer": (
            "El gas natural es un combustible fósil compuesto principalmente por metano, "
            "utilizado para generar energía y como insumo industrial."
        )
    },
    {
        "question": "¿Qué es un mineral?",
        "reference_answer": (
            "Un mineral es una sustancia natural, sólida e inorgánica con una composición "
            "química definida y una estructura cristalina."
        )
    }
]

###Evaluamos el sistema automáticamente

In [ ]:
evaluation_results = []

for item in evaluation_set:
    question = item["question"]
    reference = item["reference_answer"]

    output = answer_with_rag_complete(
        query=question,
        embedding_model=embedding_model,
        index=index,
        df_chunks=df_chunks,
        reranker=reranker,
        model=model,
        tokenizer=tokenizer,
        top_k_retrieval=8,
        top_k_final=4,
        grounding_threshold=0.55
    )

    generated = output["respuesta"]

    rouge_scores = compute_rouge_scores(reference, generated)
    bleu_score = compute_bleu_score(reference, generated)

    evaluation_results.append({
        "question": question,
        "reference_answer": reference,
        "generated_answer": generated,
        "respuesta_citada": output["respuesta_citada"],
        "grounding_score": output["grounding_score"],
        "grounding_label": output["grounding_label"],
        "is_grounded": output["is_grounded"],
        "bleu": bleu_score,
        "rouge1_f": rouge_scores["rouge1_f"],
        "rouge2_f": rouge_scores["rouge2_f"],
        "rougeL_f": rouge_scores["rougeL_f"]
    })

df_eval = pd.DataFrame(evaluation_results)
df_eval

,question,reference_answer,generated_answer,respuesta_citada,grounding_score,grounding_label,is_grounded,bleu,rouge1_f,rouge2_f,rougeL_f
0,¿Qué es la minería?,La minería es la actividad económica dedicada ...,La minería es una actividad económica del sect...,La minería es una actividad económica del sect...,0.580173,Sustentación media,True,0.035735,0.211180,0.125786,0.198758
1,¿Qué es el petróleo crudo?,El petróleo crudo es un recurso natural líquid...,El petróleo crudo es una mezcla de compuestos ...,El petróleo crudo es una mezcla de compuestos ...,0.739922,Sustentación media,True,0.055405,0.225564,0.106870,0.165414
2,¿Qué es el gas natural?,El gas natural es un combustible fósil compues...,El gas natural (GNL) es gas natural que ha sid...,El gas natural (GNL) es gas natural que ha sid...,0.763532,Alta sustentación,True,0.047594,0.197531,0.075949,0.172840
3,¿Qué es un mineral?,"Un mineral es una sustancia natural, sólida e ...",Un mineral es una sustancia natural de composi...,Un mineral es una sustancia natural de composi...,0.814577,Alta sustentación,True,0.141601,0.619718,0.492754,0.478873


###Vemos los promedios de desempeño

In [ ]:
metricas_promedio = {
    "BLEU promedio": df_eval["bleu"].mean(),
    "ROUGE-1 promedio": df_eval["rouge1_f"].mean(),
    "ROUGE-2 promedio": df_eval["rouge2_f"].mean(),
    "ROUGE-L promedio": df_eval["rougeL_f"].mean(),
    "Grounding promedio": df_eval["grounding_score"].mean()
}

metricas_promedio

{'BLEU promedio': np.float64(0.07008398097291112),
 'ROUGE-1 promedio': np.float64(0.3134983020136811),
 'ROUGE-2 promedio': np.float64(0.2003398457016649),
 'ROUGE-L promedio': np.float64(0.2539710108548002),
 'Grounding promedio': np.float64(0.7245507737000783)}

###Mostramos los resultados más claros

In [ ]:
print("=== MÉTRICAS PROMEDIO ===")
for k, v in metricas_promedio.items():
    print(f"{k}: {v:.4f}")

=== MÉTRICAS PROMEDIO ===
BLEU promedio: 0.0701
ROUGE-1 promedio: 0.3135
ROUGE-2 promedio: 0.2003
ROUGE-L promedio: 0.2540
Grounding promedio: 0.7246


###Vemos la tabla resumida

In [ ]:
df_eval[[
    "question",
    "bleu",
    "rouge1_f",
    "rouge2_f",
    "rougeL_f",
    "grounding_score",
    "grounding_label",
    "is_grounded"
]]

,question,bleu,rouge1_f,rouge2_f,rougeL_f,grounding_score,grounding_label,is_grounded
0,¿Qué es la minería?,0.035735,0.211180,0.125786,0.198758,0.580173,Sustentación media,True
1,¿Qué es el petróleo crudo?,0.055405,0.225564,0.106870,0.165414,0.739922,Sustentación media,True
2,¿Qué es el gas natural?,0.047594,0.197531,0.075949,0.172840,0.763532,Alta sustentación,True
3,¿Qué es un mineral?,0.141601,0.619718,0.492754,0.478873,0.814577,Alta sustentación,True


###Guardamos los resultados

In [ ]:
df_eval.to_csv("evaluation_results_rag.csv", index=False, encoding="utf-8")
print("Resultados de evaluación guardados.")

Resultados de evaluación guardados.


#12. BM25 + Hybrid Retrieval

In [ ]:
!pip -q install rank-bm25

In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np

###Construye BM25 sobre tus chunks

In [ ]:
tokenized_corpus = [text.lower().split() for text in df_chunks["chunk_text"].tolist()]
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 listo.")

BM25 listo.


###Construye BM25 sobre tus chunks

In [ ]:
def retrieve_chunks_bm25(query, bm25, df_chunks, top_k=5):
    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:top_k]
    resultados = df_chunks.iloc[top_indices].copy()
    resultados["bm25_score"] = [scores[i] for i in top_indices]

    return resultados

###Hybrid BM25 + FAISS

In [ ]:
def retrieve_chunks_hybrid(query, embedding_model, index, df_chunks, bm25, top_k_faiss=8, top_k_bm25=8, top_k_final=8):
    # FAISS
    faiss_results = retrieve_chunks(
        query=query,
        embedding_model=embedding_model,
        index=index,
        df_chunks=df_chunks,
        top_k=top_k_faiss
    ).copy()
    faiss_results["faiss_rank"] = range(1, len(faiss_results) + 1)

    # BM25
    bm25_results = retrieve_chunks_bm25(
        query=query,
        bm25=bm25,
        df_chunks=df_chunks,
        top_k=top_k_bm25
    ).copy()
    bm25_results["bm25_rank"] = range(1, len(bm25_results) + 1)

    # Fusion por chunk_id
    hybrid = pd.merge(
        faiss_results[["chunk_id", "title", "url", "chunk_text", "faiss_rank"]],
        bm25_results[["chunk_id", "bm25_rank"]],
        on="chunk_id",
        how="outer"
    )

    # RRF simple
    hybrid["faiss_rank"] = hybrid["faiss_rank"].fillna(1000)
    hybrid["bm25_rank"] = hybrid["bm25_rank"].fillna(1000)

    k = 60
    hybrid["hybrid_score"] = 1 / (k + hybrid["faiss_rank"]) + 1 / (k + hybrid["bm25_rank"])

    hybrid = hybrid.sort_values("hybrid_score", ascending=False).head(top_k_final).reset_index(drop=True)

    return hybrid

###Prueba rápida

In [ ]:
pregunta = "¿Qué es la minería?"

hybrid_results = retrieve_chunks_hybrid(
    query=pregunta,
    embedding_model=embedding_model,
    index=index,
    df_chunks=df_chunks,
    bm25=bm25,
    top_k_faiss=8,
    top_k_bm25=8,
    top_k_final=8
)

hybrid_results[["chunk_id", "title", "url", "faiss_rank", "bm25_rank", "hybrid_score"]]

,chunk_id,title,url,faiss_rank,bm25_rank,hybrid_score
0,DOC_0464_CHUNK_021,NaN,NaN,1000.0,1.0,0.017337
1,DOC_0482_CHUNK_001,Minero,https://es.wikipedia.org/wiki/Minero,1.0,1000.0,0.017337
2,DOC_0533_CHUNK_016,NaN,NaN,1000.0,2.0,0.017072
3,DOC_0308_CHUNK_001,Mina (minería),https://es.wikipedia.org/wiki/Mina_(miner%C3%ADa),2.0,1000.0,0.017072
4,DOC_0287_CHUNK_001,Minería,https://es.wikipedia.org/wiki/Miner%C3%ADa,3.0,1000.0,0.016816
5,DOC_0428_CHUNK_002,NaN,NaN,1000.0,3.0,0.016816
6,DOC_0416_CHUNK_001,Liberación mineral,https://es.wikipedia.org/wiki/Liberaci%C3%B3n_...,4.0,1000.0,0.016568
7,DOC_0233_CHUNK_008,NaN,NaN,1000.0,4.0,0.016568


###Pipeline final usando Hybrid + Reranker + Qwen

In [ ]:
def answer_with_rag_hybrid(
    query,
    embedding_model,
    index,
    df_chunks,
    bm25,
    reranker,
    model,
    tokenizer,
    top_k_hybrid=8,
    top_k_final=4,
    grounding_threshold=0.55
):
    # 1. Hybrid retrieval
    hybrid_results = retrieve_chunks_hybrid(
        query=query,
        embedding_model=embedding_model,
        index=index,
        df_chunks=df_chunks,
        bm25=bm25,
        top_k_faiss=8,
        top_k_bm25=8,
        top_k_final=top_k_hybrid
    )

    # 2. Rerank
    resultados_rerank = rerank_chunks(
        query=query,
        resultados_faiss=hybrid_results.rename(columns={"hybrid_score": "score_distance"}),
        reranker=reranker
    )

    resultados_finales = resultados_rerank.head(top_k_final).copy()

    # 3. Contexto
    context = build_context_from_results(resultados_finales, max_chunks=top_k_final)

    # 4. Generación
    respuesta = generate_qwen_response(query, context, model, tokenizer)

    # 5. Citation per sentence
    citas = assign_citation_per_sentence(
        response_text=respuesta,
        resultados_finales=resultados_finales,
        embedding_model=embedding_model
    )

    respuesta_citada = build_cited_response(citas)

    # 6. Grounding
    diagnostico = hallucination_guard(citas, threshold=grounding_threshold)

    return {
        "query": query,
        "respuesta": respuesta,
        "respuesta_citada": respuesta_citada,
        "citas": citas,
        "resultados_finales": resultados_finales,
        "grounding_score": diagnostico["grounding_score"],
        "grounding_label": diagnostico["label"],
        "is_grounded": diagnostico["is_grounded"],
        "grounding_warning": diagnostico["warning"]
    }

###Celda final de comparación simple

In [ ]:
pregunta = "¿Qué es la minería?"

faiss_only = retrieve_chunks(
    query=pregunta,
    embedding_model=embedding_model,
    index=index,
    df_chunks=df_chunks,
    top_k=5
)

bm25_only = retrieve_chunks_bm25(
    query=pregunta,
    bm25=bm25,
    df_chunks=df_chunks,
    top_k=5
)

hybrid_only = retrieve_chunks_hybrid(
    query=pregunta,
    embedding_model=embedding_model,
    index=index,
    df_chunks=df_chunks,
    bm25=bm25,
    top_k_faiss=8,
    top_k_bm25=8,
    top_k_final=5
)

print("=== FAISS ===")
display(faiss_only[["chunk_id", "title", "url", "score_distance"]])

print("=== BM25 ===")
display(bm25_only[["chunk_id", "title", "url", "bm25_score"]])

print("=== HYBRID BM25 + FAISS ===")
display(hybrid_only[["chunk_id", "title", "url", "faiss_rank", "bm25_rank", "hybrid_score"]])

=== FAISS ===


,chunk_id,title,url,score_distance
2416,DOC_0482_CHUNK_001,Minero,https://es.wikipedia.org/wiki/Minero,10.919002
1537,DOC_0308_CHUNK_001,Mina (minería),https://es.wikipedia.org/wiki/Mina_(miner%C3%ADa),12.925304
1448,DOC_0287_CHUNK_001,Minería,https://es.wikipedia.org/wiki/Miner%C3%ADa,13.516582
2115,DOC_0416_CHUNK_001,Liberación mineral,https://es.wikipedia.org/wiki/Liberaci%C3%B3n_...,13.779060
1511,DOC_0297_CHUNK_001,Mina a cielo abierto,https://es.wikipedia.org/wiki/Mina_a_cielo_abi...,15.637816


=== BM25 ===


,chunk_id,title,url,bm25_score
2286,DOC_0464_CHUNK_021,Arturo Umberto Illia,https://es.wikipedia.org/wiki/Arturo_Umberto_I...,13.374289
2607,DOC_0533_CHUNK_016,Patrón oro,https://es.wikipedia.org/wiki/Patr%C3%B3n_oro,12.890486
2146,DOC_0428_CHUNK_002,Historia de la industria petrolera en los Esta...,https://es.wikipedia.org/wiki/Historia_de_la_i...,12.652254
1171,DOC_0233_CHUNK_008,Organización de Países Exportadores de Petróleo,https://es.wikipedia.org/wiki/Organizaci%C3%B3...,12.403992
1170,DOC_0233_CHUNK_007,Organización de Países Exportadores de Petróleo,https://es.wikipedia.org/wiki/Organizaci%C3%B3...,12.340511


=== HYBRID BM25 + FAISS ===


,chunk_id,title,url,faiss_rank,bm25_rank,hybrid_score
0,DOC_0464_CHUNK_021,NaN,NaN,1000.0,1.0,0.017337
1,DOC_0482_CHUNK_001,Minero,https://es.wikipedia.org/wiki/Minero,1.0,1000.0,0.017337
2,DOC_0533_CHUNK_016,NaN,NaN,1000.0,2.0,0.017072
3,DOC_0308_CHUNK_001,Mina (minería),https://es.wikipedia.org/wiki/Mina_(miner%C3%ADa),2.0,1000.0,0.017072
4,DOC_0287_CHUNK_001,Minería,https://es.wikipedia.org/wiki/Miner%C3%ADa,3.0,1000.0,0.016816
